In [2]:
%tb
import os, json, math, random, glob
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from collections import defaultdict
from modules.Plotting import MetricLog, plot_metrics
from modules.HandTesting import hand_test_repl

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from transformers import T5ForConditionalGeneration, AutoTokenizer
from torch.optim import AdamW

import warnings
warnings.filterwarnings("ignore")


WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")

LINE_MODEL_NAME = "line_model_T5_small"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

ModuleNotFoundError: No module named 'modules'

WORKDIR: C:\Users\Roman\Documents\Projects\code_autocomplete
[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1


## Tokenizer

In [3]:
# character-level BPE-lite

SPECIAL = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}

class CodeTokenizer:
    """
    Simple sub-word tokenizer tailored for Python source code.
    Splits on whitespace/punctuation, keeps indentation tokens,
    and falls back to characters for unknowns.
    """
    PUNCT = set(",;&|~^@#")

    def __init__(self, vocab_size: int = 8000):
        self.vocab_size = vocab_size
        self.token2id: Dict[str, int] = dict(SPECIAL)
        self.id2token: Dict[int, str] = {v: k for k, v in SPECIAL.items()}
        self.built = False

    # ── build ──────────────────────────────────────────────
    def build(self, texts: List[str], min_freq: int = 3):
        freq: Dict[str, int] = defaultdict(int)
        for t in texts:
            for tok in self._raw_split(t):
                freq[tok] += 1
        sorted_tokens = sorted(freq.items(), key=lambda x: -x[1])
        for tok, cnt in sorted_tokens:
            if cnt < min_freq:
                break
            if tok not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[tok] = idx
                self.id2token[idx] = tok
        # fill remaining slots with single chars
        for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,_ \t\n":  #for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_ \t\n":
            if c not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[c] = idx
                self.id2token[idx] = c
        self.built = True
        print(f"[Tokenizer] vocab_size={len(self.token2id)}")

    def _raw_split(self, text: str) -> List[str]:
        tokens = []
        for line in text.splitlines(keepends=True):
            # capture leading whitespace as indent token
            stripped = line.lstrip(" \t")
            indent = line[: len(line) - len(stripped)]
            for ch in indent:
                tokens.append(ch)
            # split remainder on punctuation / spaces
            buf = ""
            for ch in stripped:
                if ch in self.PUNCT or ch in " \t\n\r":
                    if ch == "\n" or ch.strip():
                        if buf:
                            tokens.append(buf)
                        tokens.append(ch)
                        continue
                    if buf:
                        buf += ch
                        tokens.append(buf)
                        buf = ""
                        tokens.append("\n")
                else:
                    buf += ch
            if buf:
                tokens.append(buf)
        return tokens

    def encode(self, text: str) -> List[int]:
        ids = [SPECIAL["<BOS>"]]
        for tok in self._raw_split(text):
            if tok in self.token2id:
                ids.append(self.token2id[tok])
            else:
                # char fallback
                for ch in tok:
                    ids.append(self.token2id.get(ch, SPECIAL["<UNK>"]))
        ids.append(SPECIAL["<EOS>"])
        return ids

    def decode(self, ids: List[int]) -> str:
        parts = []
        for i in ids:
            tok = self.id2token.get(i, "")
            if tok in SPECIAL:
                continue
            parts.append(tok)
        return "".join(parts)

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump({"token2id": self.token2id}, f)

    @classmethod
    def load(cls, path: str) -> "CodeTokenizer":
        with open(path) as f:
            d = json.load(f)
        obj = cls()
        obj.token2id = {k: int(v) for k, v in d["token2id"].items()}
        obj.id2token = {v: k for k, v in obj.token2id.items()}
        obj.built = True
        return obj

    @property
    def pad_id(self):  return SPECIAL["<PAD>"]
    @property
    def eos_id(self):  return SPECIAL["<EOS>"]
    @property
    def bos_id(self):  return SPECIAL["<BOS>"]
    @property
    def vocab(self):   return len(self.token2id)

## Datasets

In [4]:

def load_files(data_dir: str, max_files: int = 0) -> List[str]:
    """Load .py / .txt files from a directory tree."""
    patterns = ["**/*.py", "**/*.txt"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(data_dir, pat), recursive=True))
    if max_files:
        files = files[:max_files]
    texts = []
    for fp in files:
        try:
            texts.append(Path(fp).read_text(errors="replace"))
        except Exception:
            pass
    print(f"[Data] loaded {len(texts)} files from {data_dir}")
    return texts



class TokenDataset(Dataset):
    """
    Sliding-window dataset for next-token prediction.
    Target at each position is the next token id.
    """
    def __init__(self, ids: List[int], ctx: int = 128):
        self.ctx = ctx
        self.data = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return max(0, len(self.data) - self.ctx - 1)

    def __getitem__(self, i):
        x = self.data[i: i + self.ctx]
        y = self.data[i + 1: i + self.ctx + 1]
        return x, y


class T5LineDataset(Dataset):
    """
    Each sample: tokenise the prefix with AutoTokenizer → input_ids
                 tokenise the suffix                        → labels
    Padding and label-masking are handled in the collator below.
    """
    def __init__(self, texts: List[str], hf_tokenizer: AutoTokenizer,
                 max_prefix: int = 96, max_suffix: int = 64):
        self.tok = hf_tokenizer
        self.max_prefix = max_prefix
        self.max_suffix = max_suffix
        self.samples: List[Tuple[str, str]] = []

        for text in texts:
            for line in text.splitlines():
                line = line.rstrip()
                if len(line.strip()) < 10:
                    continue
                # split at 30–70 % of the line (your "root cause 2" fix)
                words = line.split()
                if len(words) < 3:
                    continue
                cut = random.randint(
                    max(1, int(len(words) * 0.3)),
                    max(2, int(len(words) * 0.7)),
                )
                prefix = " ".join(words[:cut])
                suffix = " ".join(words[cut:])
                self.samples.append((prefix, suffix))

        print(f"[T5LineDataset] {len(self.samples)} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        prefix, suffix = self.samples[i]
        enc = self.tok(
            prefix,
            max_length=self.max_prefix,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        dec = self.tok(
            suffix,
            max_length=self.max_suffix,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            dec["input_ids"].squeeze(0),
        )



def collate_t5(batch, pad_id: int):
    """Pad input_ids, attention_mask, and labels in a single pass."""
    src_ids, src_masks, lbl_ids = zip(*batch)

    max_src = max(t.size(0) for t in src_ids)
    max_lbl = max(t.size(0) for t in lbl_ids)

    B = len(batch)
    SRC  = torch.full((B, max_src), pad_id, dtype=torch.long)
    MASK = torch.zeros((B, max_src), dtype=torch.long)
    LBL  = torch.full((B, max_lbl), -100,   dtype=torch.long)   # -100 = ignored by T5 loss

    for i, (s, m, l) in enumerate(zip(src_ids, src_masks, lbl_ids)):
        SRC[i,  :s.size(0)] = s
        MASK[i, :m.size(0)] = m
        LBL[i,  :l.size(0)] = l

    return SRC, MASK, LBL

## Models

In [5]:
@dataclass
class ModelCfg:
    vocab: int = 8000
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    d_ff: int = 1024
    max_len: int = 256
    dropout: float = 0.1


class PositionalEncoding(nn.Module):
    def __init__(self, d: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class TokenModel(nn.Module):
    """
    Decoder-only Transformer for causal next-token prediction.
    """
    def __init__(self, cfg: ModelCfg):
        super().__init__()
        self.cfg = cfg
        self.emb   = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.pos   = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        layer      = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout,
            batch_first=True, norm_first=True
        )
        self.enc   = nn.TransformerEncoder(layer, cfg.n_layers)
        self.head  = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.emb.weight = self.head.weight  # weight tying

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        h = self.pos(self.emb(x))
        h = self.enc(h, mask=mask, is_causal=True)
        return self.head(h)

    @torch.no_grad()
    def generate(self, prefix_ids: List[int], max_new: int,
                 temperature: float = 0.8, top_k: int = 50,
                 stop_at_word_end: bool = True,
                 tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
        self.eval()
        dev   = next(self.parameters()).device
        ids   = list(prefix_ids)
        generated = []
        PUNCT_CHARS = set("()[]{}.,;:=+-*/\\%<>!&|~^@# \t\n\"'`")
        for _ in range(max_new):
            x = torch.tensor([ids[-self.cfg.max_len:]], dtype=torch.long, device=dev)
            logits = self(x)[0, -1] / temperature
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == SPECIAL["<EOS>"]:
                break
            ids.append(nxt)
            generated.append(nxt)
            if stop_at_word_end and tokenizer:
                tok = tokenizer.id2token.get(nxt, "")
                if any(c in PUNCT_CHARS for c in tok):
                    break
        return generated

# Best model saving

In [6]:
import os
from pathlib import Path
import torch
from torch import nn
from typing import List, Tuple, Optional

class BestModelSaver:
    """Keeps the best N checkpoints by val loss."""
    def __init__(self, ckpt_dir: str, model_name: str, keep: int = 3):
        self.dir   = Path(ckpt_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.name  = model_name
        self.keep  = keep
        self.saved: List[Tuple[float, str]] = []  # (val_loss, path)

    def save(self, model: nn.Module, val_loss: float, epoch: int, extra: dict = None):
        path = str(self.dir / f"{self.name}_ep{epoch:03d}_loss{val_loss:.4f}.pt")
        payload = {
            "model_state": model.state_dict(),
            "val_loss":    val_loss,
            "epoch":       epoch,
            "cfg":         model.cfg,
            # "cfg":         getattr(model, "cfg", None),
        }
        if extra:
            payload.update(extra)
        torch.save(payload, path)
        self.saved.append((val_loss, path))
        self.saved.sort(key=lambda x: x[0])
        while len(self.saved) > self.keep:
            _, old = self.saved.pop()
            try:   os.remove(old)
            except FileNotFoundError: pass
            print(f"[Saver] removed old ckpt: {old}")
        print(f"[Saver] saved ckpt: {path}  (val_loss={val_loss:.4f})")

    def best_path(self) -> Optional[str]:
        return self.saved[0][1] if self.saved else None

## Training loop

In [7]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_line_model(
    model:    T5ForConditionalGeneration, 
    hf_tok:   AutoTokenizer,             # HuggingFace tokenizer
    train_dl: DataLoader,
    val_dl:   DataLoader,
    epochs:   int,
    lr:       float,
    device:   torch.device,
    saver:    BestModelSaver,
    log:      MetricLog,
    plot_dir: str,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    # T5 has its own internal cross-entropy; we just call model(...).loss
    opt   = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)

    for ep in range(1, epochs + 1):
        # ── TRAIN ────────────────────────────────────────────────────────────
        model.train()
        t_loss = t_acc = t_steps = 0
        gn = 0.0

        batch_bar = tqdm(train_dl,
                         desc=f"[Line] Epoch {ep}/{epochs} train",
                         leave=False, unit="batch")

        for src, mask, lbl in batch_bar:
            src, mask, lbl = src.to(device), mask.to(device), lbl.to(device)

            out  = model(input_ids=src, attention_mask=mask, labels=lbl)
            loss = out.loss          # T5 computes CE internally, -100 labels ignored

            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()

            # token accuracy: compare argmax logits vs labels where label != -100
            with torch.no_grad():
                preds = out.logits.argmax(-1)
                valid = (lbl != -100)
                acc   = (preds[valid] == lbl[valid]).float().mean().item()

            t_loss  += loss.item()
            t_acc   += acc
            t_steps += 1

            batch_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.3f}")

        tl = t_loss / t_steps
        ta = t_acc  / t_steps

        # ── VAL ──────────────────────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0

        with torch.no_grad():
            for src, mask, lbl in tqdm(val_dl,
                                       desc=f"[Line] Epoch {ep}/{epochs} val  ",
                                       leave=False, unit="batch"):
                src, mask, lbl = src.to(device), mask.to(device), lbl.to(device)
                out    = model(input_ids=src, attention_mask=mask, labels=lbl)
                v_loss  += out.loss.item()
                v_steps += 1

        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        # ── LOGGING ──────────────────────────────────────────────────────────
        log.append(
            train_loss=tl, val_loss=vl,
            train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
            lr=opt.param_groups[0]["lr"],
            token_acc=ta, grad_norm=gn,
        )
        tqdm.write(
            f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        # save best checkpoint — store HF model with torch.save so BestModelSaver
        # stays unchanged; pass the raw state dict the same way
        saver.save(model, vl, ep)

        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}/{LINE_MODEL_NAME}_ep{ep:02d}.png")

    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final", f"{plot_dir}/{LINE_MODEL_NAME}/{LINE_MODEL_NAME}_final.png")

## Main

In [8]:
class Arguments():
    def __init__(self, data_dir: str = f"{WORKDIR}/Clean_Dataset", ckpt_dir: str = f"{WORKDIR}/checkpoints",
                    plot_dir: str = F"{WORKDIR}/plots", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    args = Arguments(epochs=10)
    # args = Arguments(max_files=100)
    # args = Arguments(skip_line=True, max_files=100, epochs=1)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    # args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads,  n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )

    torch.serialization.add_safe_globals([ModelCfg])

    HF_MODEL = "Salesforce/codet5-small"
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        lm = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / LINE_MODEL_NAME / f"{LINE_MODEL_NAME}_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
    
        hand_test_repl(tm, lm, tokenizer, hf_tok, device)
        return
    

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]

    # ── LINE MODEL ───────────────────────────────────────────
    if not args.skip_line:
        print("  Prepairing LINE model")

        line_model = T5ForConditionalGeneration.from_pretrained(HF_MODEL).to(device)
        
        collate    = lambda b: collate_t5(b, hf_tok.pad_token_id)
        tr_line_ds = T5LineDataset(tr_txt, hf_tok)
        va_line_ds = T5LineDataset(va_txt, hf_tok)
        tr_line_dl = DataLoader(tr_line_ds, args.batch, shuffle=True,
                                collate_fn=collate, num_workers=0, pin_memory=True)
        va_line_dl = DataLoader(va_line_ds, args.batch, shuffle=False,
                                collate_fn=collate, num_workers=0, pin_memory=True)
        
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.1f}M parameters (codet5-small)")
        
        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
        line_log   = MetricLog()
        train_line_model(line_model, hf_tok, tr_line_dl, va_line_dl,
                         args.epochs, args.lr, device, line_saver, line_log, args.plot_dir)

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(None, line_model, tokenizer, hf_tok, device)


main()

[Tokenizer] loading tokenizer.json
[Loading] Started loading


KeyboardInterrupt: 